In [0]:
----------------------------------------
CREATE OR REPLACE TABLE `capgemini_academy`.`gold`.`dim_datas` ( 
	dat_sk_data BIGINT NOT NULL COMMENT 'Primary key e Surrogate key da tabela',
	dat_data DATE NOT NULL COMMENT 'Data no formato YYYY-MM-DD',
	dat_nu_dia SMALLINT NOT NULL COMMENT 'Número do dia no mês',
	dat_nu_mes SMALLINT NOT NULL COMMENT 'Número do mês',
	dat_nu_ano SMALLINT NOT NULL COMMENT 'Ano',
	dat_nu_trimestre SMALLINT NOT NULL COMMENT 'Número do trimestre',
	dat_nm_dia_semana STRING NOT NULL COMMENT 'Nome do dia na semana',
	dat_dh_atualizacao TIMESTAMP DEFAULT from_utc_timestamp(now(), 'GMT-3') COMMENT 'Data/hora da última atualização da linha',
	    CONSTRAINT pk_dim_datas PRIMARY KEY ( dat_sk_data )
 ) USING DELTA
    TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported',
                    'delta.logRetentionDuration'='7 days',
					'delta.autoOptimize.autoCompact'='auto',
                    'spark.databricks.delta.autoCompact.enabled'='true');

COMMENT ON TABLE `capgemini_academy`.`gold`.`dim_datas` IS 'A tabela [dim_datas] contém informações sobre datas, incluindo ano, mês, dia, dia da semana. Esta tabela é útil para análise baseada em tempo e pode ser unida a outras tabelas para fornecer contexto para eventos de negócios que ocorreram em datas específicas.';
----------------------------------------
INSERT INTO `capgemini_academy`.`gold`.`dim_datas` (dat_sk_data, dat_data, dat_nu_dia, dat_nu_mes, dat_nu_ano, dat_nu_trimestre, dat_nm_dia_semana)
    WITH date_range AS (
        SELECT sequence(
            DATE('2024-01-01'),
            DATE('2026-12-31'),
            INTERVAL 1 DAY
        ) AS dates
    ),
    expanded AS (
        SELECT EXPLODE(dates) AS dat_data
        FROM date_range
    )

SELECT
    CAST(DATE_FORMAT(dat_data, 'yyyyMMdd') AS BIGINT) AS dat_sk_data,
    dat_data,
    CAST(DAY(dat_data) AS SMALLINT) AS dat_nu_dia,
    CAST(MONTH(dat_data) AS SMALLINT) AS dat_nu_mes,
    CAST(YEAR(dat_data) AS SMALLINT) AS dat_nu_ano,
    CAST(QUARTER(dat_data) AS SMALLINT) AS dat_nu_trimestre,
    CASE DAYOFWEEK(dat_data)
        WHEN 1 THEN 'Domingo'
        WHEN 2 THEN 'Segunda-feira'
        WHEN 3 THEN 'Terça-feira'
        WHEN 4 THEN 'Quarta-feira'
        WHEN 5 THEN 'Quinta-feira'
        WHEN 6 THEN 'Sexta-feira'
        WHEN 7 THEN 'Sábado'
    END AS dat_nm_dia_semana
FROM expanded
ORDER BY dat_data;
----------------------------------------
